# GDELT News Headlines — WTI Crude Oil

This notebook queries the GDELT Project API to retrieve news article headlines 
related to WTI crude oil prices over a two-year period (2024–2026). Articles are 
downloaded week by week to avoid API rate limiting. The raw dataset is then cleaned 
by filtering to English-language articles only, removing duplicates, and parsing 
timestamps. The cleaned dataset contains article titles, publication timestamps, 
source domains, and URLs, and is saved to 
`01_data/processed/gdelt_wti_clean.csv`.

### General imports

In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

### Main search function

Adjusts request headers so that instead of identifying as Python, requests are sent with a Mozilla browser User-Agent string. This avoids being blocked by GDELT's bot detection.

**Note:** GDELT API results are capped at 250 articles per request. Weeks where the download returns exactly 250 articles may be truncated. This affects a small number of high-activity periods and is considered acceptable for the purposes of this study.

In [2]:
def search_gdelt(query, start_date, end_date, max_records=250):
    url = "https://api.gdeltproject.org/api/v2/doc/doc"
    params = {
        "query": query,
        "mode": "artlist",
        "maxrecords": max_records,
        "startdatetime": start_date.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end_date.strftime("%Y%m%d%H%M%S"),
        "format": "json"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers, timeout=30)
    return response.json()

### Download by date range

Executes the search function iteratively over a range of weeks between `start_date` and `end_date`. A cooldown between requests is included to reduce the number of rejections caused by GDELT's rate limiting.

In [ ]:
def download_gdelt_range(query, start_date, end_date, sleep_seconds=8):
    all_articles = []
    current = start_date
    
    while current < end_date:
        week_end = min(current + timedelta(days=7), end_date)
        try:
            results = search_gdelt(query, current, week_end)
            articles = results.get("articles", [])
            for a in articles:
                a['week_start'] = current.strftime("%Y-%m-%d")
            all_articles.extend(articles)
            print(f"{current.date()} → {week_end.date()}: {len(articles)} articles")
        except Exception as e:
            print(f"{current.date()} → {week_end.date()}: ERROR — {str(e)[:50]}")
        
        current = week_end
        time.sleep(sleep_seconds)
    
    return pd.DataFrame(all_articles)

### Download weekly headlines

GDELT does not provide article body text through its API — only metadata including title, URL, publication date, domain, and language. Body text is retrieved separately in `04_gdelt_scraper.ipynb` by scraping each article URL individually.

In [6]:
df_news = download_gdelt_range(
    query="crude oil WTI price",
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2026, 3, 1)
)

2024-01-01 → 2024-01-08: ERROR — Expecting value: line 1 column 1 (char 0)
2024-01-08 → 2024-01-15: ERROR — Expecting value: line 1 column 1 (char 0)
2024-01-15 → 2024-01-22: 112 artículos
2024-01-22 → 2024-01-29: 102 artículos
2024-01-29 → 2024-02-05: 113 artículos
2024-02-05 → 2024-02-12: ERROR — Expecting value: line 1 column 1 (char 0)
2024-02-12 → 2024-02-19: 108 artículos
2024-02-19 → 2024-02-26: ERROR — Expecting value: line 1 column 1 (char 0)
2024-02-26 → 2024-03-04: ERROR — Expecting value: line 1 column 1 (char 0)
2024-03-04 → 2024-03-11: ERROR — Expecting value: line 1 column 1 (char 0)
2024-03-11 → 2024-03-18: ERROR — Expecting value: line 1 column 1 (char 0)
2024-03-18 → 2024-03-25: ERROR — Expecting value: line 1 column 1 (char 0)
2024-03-25 → 2024-04-01: ERROR — Expecting value: line 1 column 1 (char 0)
2024-04-01 → 2024-04-08: 195 artículos
2024-04-08 → 2024-04-15: ERROR — Expecting value: line 1 column 1 (char 0)
2024-04-15 → 2024-04-22: ERROR — HTTPSConnectionPool(ho

### Summary and save data

In [ ]:
print(f"Total articles: {len(df_news)}")
print(f"Weeks covered: {df_news['week_start'].nunique()}")
print(f"\nLanguages:\n{df_news['language'].value_counts().head()}")
print(f"\nTop domains:\n{df_news['domain'].value_counts().head(10)}")
print(df_news[['title', 'seendate', 'domain']].head(5))

filename = "gdelt_wti_raw.csv"
df_news.to_csv(f"../01_data/raw/news/{filename}", index=False)
print(f"Saved to 01_data/raw/news/{filename}")

Total artículos: 4831
Semanas cubiertas: 44

Idiomas:
language
English      3913
Spanish       435
Polish        106
Hungarian     102
Dutch          83
Name: count, dtype: int64

Dominios más frecuentes:
domain
marketscreener.com              368
zerohedge.com                   201
fxstreet.com                    185
finance.yahoo.com               137
oilprice.com                    111
fool.com.au                     109
seekingalpha.com                102
markets.financialcontent.com     74
hellenicshippingnews.com         70
theglobeandmail.com              68
Name: count, dtype: int64
                                               title          seendate  \
0  Crude Oil Weekly Forecast - 21 / 01 : Stable R...  20240121T123000Z   
1  Crude Oil struggles to pick a direction as geo...  20240115T220000Z   
2  Oil sinks with depressed OPEC report and Kuwai...  20240117T163000Z   
3  Oil in the green with Black Sea loading issues...  20240119T160000Z   
4  WTI dips to $70 . 60 and rebo

### News cleaning

The NLP models used in this project (FinBERT) are trained primarily on English financial text. Non-English articles are therefore excluded to maximize sentiment classification accuracy. The following steps are applied:

- **Language filter:** Only English-language articles are retained.
- **Deduplication:** Articles with identical titles are removed, keeping the first occurrence.
- **Timestamp parsing:** The `seendate` field is converted from GDELT's raw format (`YYYYMMDDTHHmmSSZ`) to a standard UTC datetime.
- **Column selection:** Only the columns relevant for downstream modeling are kept: `title`, `datetime`, `domain`, and `url`.

In [ ]:
df_clean = df_news.copy()

# Keep English articles only
df_clean = df_clean[df_clean['language'] == 'English']

# Remove duplicate titles
df_clean = df_clean.drop_duplicates(subset='title')

# Parse publication timestamp to UTC datetime
df_clean['datetime'] = pd.to_datetime(df_clean['seendate'], format='%Y%m%dT%H%M%SZ', utc=True)
df_clean = df_clean.sort_values('datetime').reset_index(drop=True)

# Keep only relevant columns
df_clean = df_clean[['title', 'datetime', 'domain', 'url']]

print(f"Articles after cleaning: {len(df_clean)}")
print(f"Range: {df_clean['datetime'].min()} to {df_clean['datetime'].max()}")
print(df_clean.head(5))

filename = "gdelt_wti_clean.csv"
df_clean.to_csv(f"../01_data/processed/{filename}", index=False)
print(f"Saved to 01_data/processed/{filename}")

Artículos después de limpieza: 3290
Rango: 2024-01-01 00:30:00+00:00 a 2026-02-22 11:15:00+00:00
                                               title  \
0                2023 Was a Bad Year for Commodities   
1  Global oil market  poised for significant shif...   
2        5 things to watch on the ASX 200 on Tuesday   
3  Asia Shares Get 2024 to Steady Start With Busy...   
4  oil prices : Oil jumps 1 % in New Year after U...   

                   datetime                        domain  \
0 2024-01-01 00:30:00+00:00                  oilprice.com   
1 2024-01-01 05:00:00+00:00                     zawya.com   
2 2024-01-01 22:00:00+00:00                   fool.com.au   
3 2024-01-02 04:00:00+00:00              money.usnews.com   
4 2024-01-02 04:00:00+00:00  economictimes.indiatimes.com   

                                                 url  
0  https://oilprice.com/Energy/Energy-General/202...  
1  https://www.zawya.com/en/markets/commodities/g...  
2  https://www.fool.com.au/2024/01